In [ ]:
# ============================================================
# SISTEM DST-TB — STREAMLIT DI GOOGLE COLAB
# ============================================================

# ── Install semua yang dibutuhkan ────────────────────────────
!pip install streamlit lightgbm scikit-learn pandas joblib \
             pyngrok -q

# ── Upload file model ─────────────────────────────────────────
from google.colab import files
import os

print("Upload model_GNN_LightGBM_DST.pkl ...")
uploaded = files.upload()

print("Upload scaler_DST.pkl ...")
uploaded = files.upload()

print("Upload dominant_ALL.csv ...")
uploaded = files.upload()

print("Upload drug_embeddings_GNN_v2.csv ...")
uploaded = files.upload()

print("\n✓ Semua file berhasil diupload!")
print("Files:", os.listdir('.'))

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 74.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 103.0 MB/s eta 0:00:00
Upload model_GNN_LightGBM_DST.pkl ...


Saving model_GNN_LightGBM_DST.pkl to model_GNN_LightGBM_DST.pkl
Upload scaler_DST.pkl ...


Saving scaler_DST.pkl to scaler_DST.pkl
Upload dominant_ALL.csv ...


Saving dominant_ALL.csv to dominant_ALL.csv
Upload drug_embeddings_GNN_v2.csv ...


Saving drug_embeddings_GNN_v2.csv to drug_embeddings_GNN_v2.csv

✓ Semua file berhasil diupload!
Files: ['.config', 'model_GNN_LightGBM_DST.pkl', 'dominant_ALL.csv', 'drug_embeddings_GNN_v2.csv', 'scaler_DST.pkl', 'sample_data']


In [ ]:
# ── Buat file app.py (aplikasi Streamlit) ────────────────────
app_code = '''
import streamlit as st
import pandas as pd
import numpy as np
import joblib

# ── Load model & data ─────────────────────────────────────────
@st.cache_resource
def load_assets():
    model  = joblib.load("model_GNN_LightGBM_DST.pkl")
    scaler = joblib.load("scaler_DST.pkl")
    df_ref = pd.read_csv("dominant_ALL.csv")
    df_emb = pd.read_csv("drug_embeddings_GNN_v2.csv")
    return model, scaler, df_ref, df_emb

model, scaler, df_ref, df_emb = load_assets()

gnn_cols     = [f"GNN_emb_{i+1}" for i in range(32)]
genomic_cols = ["Sensitivity_num", "Specificity_num", "PPV_num",
                "Present_pheno_R", "Present_pheno_S", "Dominance_Score"]
DRUGS        = ["RIF", "INH", "EMB", "PZA"]

# ── Fungsi prediksi DRI ───────────────────────────────────────
def prediksi_dri(profil_mutasi):
    hasil = {}
    for drug, mutasi_list in profil_mutasi.items():
        emb_row    = df_emb[df_emb["Drug"] == drug]
        gnn_values = emb_row[gnn_cols].values[0]

        if not mutasi_list:
            hasil[drug] = {"DRI": 0.0, "Status": "Sensitif", "Detail": []}
            continue

        detail = []
        for mutasi in mutasi_list:
            row = df_ref[(df_ref["Drug"] == drug) &
                         (df_ref["Variant"] == mutasi)]
            if row.empty:
                continue
            genomic_vals = row[genomic_cols].values[0]
            fitur        = np.concatenate([genomic_vals, gnn_values])
            fitur_scaled = scaler.transform(fitur.reshape(1, -1))
            prob         = float(model.predict_proba(fitur_scaled)[0][1])
            detail.append({"variant": mutasi, "dri": round(prob, 4)})

        if not detail:
            hasil[drug] = {"DRI": 0.0, "Status": "Tidak dapat diprediksi", "Detail": []}
            continue

        dri_final = max(d["dri"] for d in detail)
        if dri_final >= 0.7:
            status = "Resisten"
        elif dri_final >= 0.4:
            status = "Borderline"
        else:
            status = "Sensitif"

        hasil[drug] = {"DRI": dri_final, "Status": status, "Detail": detail}

    # Kesimpulan MDR-TB
    resisten = [d for d, v in hasil.items() if v["Status"] == "Resisten"]
    if "RIF" in resisten and "INH" in resisten:
        kesimpulan  = "🚨 MDR-TB Terdeteksi"
        rekomendasi = ("Pasien resisten terhadap Rifampisin dan Isoniazid. "
                       "Disarankan beralih ke regimen pengobatan lini kedua "
                       "sesuai panduan WHO.")
        warna_kes   = "error"
    elif resisten:
        kesimpulan  = f"⚠️ Resistensi Parsial ({', '.join(resisten)})"
        rekomendasi = ("Ditemukan resistensi pada satu atau lebih obat lini pertama. "
                       "Pertimbangkan penyesuaian regimen terapi.")
        warna_kes   = "warning"
    else:
        kesimpulan  = "✅ TB Sensitif Obat"
        rekomendasi = ("Tidak ditemukan mutasi resisten. Regimen standar lini "
                       "pertama (HRZE) dapat dilanjutkan.")
        warna_kes   = "success"

    return hasil, kesimpulan, rekomendasi, warna_kes

# ── UI Streamlit ──────────────────────────────────────────────
st.set_page_config(
    page_title = "Sistem DST-TB",
    page_icon  = "🧬",
    layout     = "wide"
)

# Header
st.markdown("## 🧬 Sistem DST-TB Berbasis GNN-LightGBM")
st.markdown("**Drug Susceptibility Testing — Prediksi Resistensi MDR-TB**")
st.markdown("*Input profil mutasi dari hasil TCM / GeneXpert*")
st.divider()

# Input mutasi per obat
st.markdown("### Input Profil Mutasi Pasien")

col1, col2 = st.columns(2)
profil_mutasi = {}

warna_obat = {"RIF": "🔴", "INH": "🟢", "EMB": "🔵", "PZA": "🟡"}

for i, drug in enumerate(DRUGS):
    col = col1 if i % 2 == 0 else col2
    with col:
        variants_tersedia = df_ref[df_ref["Drug"] == drug]["Variant"].tolist()
        dipilih = st.multiselect(
            f"{warna_obat[drug]} {drug} — Pilih mutasi yang terdeteksi:",
            options   = variants_tersedia,
            key       = f"select_{drug}",
            help      = f"Pilih mutasi {drug} yang ditemukan dari hasil TCM/GeneXpert"
        )
        profil_mutasi[drug] = dipilih

st.divider()

# Tombol prediksi
col_btn1, col_btn2, _ = st.columns([1, 1, 4])
with col_btn1:
    prediksi_btn = st.button("🔬 Prediksi DRI", type="primary", use_container_width=True)
with col_btn2:
    reset_btn = st.button("🔄 Reset", use_container_width=True)

if reset_btn:
    st.rerun()

# Hasil prediksi
if prediksi_btn:
    with st.spinner("Memproses prediksi..."):
        hasil, kesimpulan, rekomendasi, warna_kes = prediksi_dri(profil_mutasi)

    st.markdown("### Hasil Prediksi DRI")

    # Kartu DRI per obat
    cols = st.columns(4)
    status_emoji = {"Resisten": "🔴", "Borderline": "🟡",
                    "Sensitif": "🟢", "Tidak dapat diprediksi": "⚫"}
    status_color = {"Resisten": "#C62828", "Borderline": "#E65100",
                    "Sensitif": "#2E7D32", "Tidak dapat diprediksi": "#424242"}
    status_text  = {"Resisten": "#FFFFFF", "Borderline": "#FFFFFF",
                    "Sensitif": "#FFFFFF", "Tidak dapat diprediksi": "#FFFFFF"}

    for col, drug in zip(cols, DRUGS):
        res = hasil.get(drug, {"DRI": 0.0, "Status": "Sensitif", "Detail": []})
        with col:
            st.markdown(
                f"""<div style="background:{status_color[res['Status']]};
                    border-radius:10px; padding:16px; text-align:center;
                    border:1px solid rgba(255,255,255,0.1);">
                    <h3 style="margin:0; color:#FFFFFF">{drug}</h3>
                    <h1 style="margin:8px 0; color:#FFFFFF">{res['DRI']:.2f}</h1>
                    <p style="margin:0; font-weight:bold; color:#FFFFFF">
                        {status_emoji[res['Status']]} {res['Status']}
                    </p>
                </div>""",
                unsafe_allow_html=True
            )
            # Detail mutasi
            if res["Detail"]:
                with st.expander("Detail mutasi"):
                    for d in res["Detail"]:
                        st.write(f"• `{d['variant']}` → DRI: **{d['dri']:.4f}**")

    st.divider()

    # Kesimpulan & Rekomendasi
    st.markdown("### Kesimpulan Klinis")
    if warna_kes == "error":
        st.error(f"**{kesimpulan}**")
    elif warna_kes == "warning":
        st.warning(f"**{kesimpulan}**")
    else:
        st.success(f"**{kesimpulan}**")

    st.info(f"**Rekomendasi:** {rekomendasi}")
    st.caption("*Hasil prediksi bersifat pendukung keputusan klinis. "
               "Konfirmasi oleh tenaga medis tetap diperlukan.*")

    # Tabel ringkasan
    st.divider()
    st.markdown("### Ringkasan Hasil")
    df_ringkasan = pd.DataFrame([
        {"Obat": drug,
         "DRI": f"{hasil[drug]['DRI']:.4f}",
         "Status": hasil[drug]["Status"],
         "Mutasi Terdeteksi": ", ".join(
             [d["variant"] for d in hasil[drug]["Detail"]]
         ) or "Tidak ada"}
        for drug in DRUGS
    ])
    st.dataframe(df_ringkasan, use_container_width=True, hide_index=True)
'''

# Simpan ke file app.py
with open("app.py", "w") as f:
    f.write(app_code)

print("✓ File app.py berhasil dibuat!")

✓ File app.py berhasil dibuat!


In [ ]:
# ── Jalankan Streamlit dengan ngrok ──────────────────────────
from pyngrok import ngrok
import subprocess
import time

# Daftar gratis di https://ngrok.com → My Authtoken → copy token
# Paste token kamu di bawah ini:
ngrok.set_auth_token("3Fd7Yab7YTysTFUiVcx7R15JsYq_5JQBS2KSR4X2gAWwHgge1")

# Jalankan streamlit di background
proc = subprocess.Popen(
    ["python", "-m", "streamlit", "run", "app.py",
     "--server.port", "8501",
     "--server.headless", "true"],
    stdout = subprocess.PIPE,
    stderr = subprocess.PIPE
)

time.sleep(8)  # tunggu streamlit siap

# Buat tunnel
public_url = ngrok.connect(8501)
print("="*50)
print(f"✅ SISTEM DST-TB BERJALAN!")
print(f"🌐 Akses di: {public_url}")
print("="*50)

✅ SISTEM DST-TB BERJALAN!
🌐 Akses di: NgrokTunnel: "https://duchess-sacrifice-gainfully.ngrok-free.dev" -> "http://localhost:8501"
